In [ ]:
# Setup cell. Canonical text: paste this file into the first code cell of every stage notebook.
STAGE = "4b"  # the only line that differs between notebooks

import contextlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
OBJECT = "my-object"  # folder name under real-to-sim-twin/ for your capture
RECORDING_ID = "real-to-sim-twin"
REFERENCE_URL = "https://artifacts.aegeanai.com/projects/real-to-sim-twin/reference"

RERUN = "rerun-sdk[notebook]==0.41.0"
PACKAGES = {
    1: ["pycolmap==4.2.1"],
    2: ["git+https://github.com/facebookresearch/sam2@2b90b9f", "huggingface_hub", "gsplat==1.5.3", "ninja",
        "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    3: ["gsplat==1.5.3", "ninja", "open3d==0.20.0", "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    4: ["diffusers==0.42.1", "transformers==5.21.0", "xatlas==0.0.11", "open3d==0.20.0", "pycolmap==4.2.1"],
    "4b": ["lpips==0.1.4", "pycolmap==4.2.1", "ninja"],
    5: ["trimesh==5.2.0", "coacd==1.0.14"],
    6: ["newton[sim,importers]==1.7.0", "usd-core==26.8", "trimesh==5.2.0", "lpips==0.1.4", "gsplat==1.5.3", "ninja",
        "pycolmap==4.2.1", "open3d==0.20.0"],
}
# Built against the installed torch, so pip must not isolate the build.
NO_BUILD_ISOLATION = {"4b": ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"],
                      6: ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"]}
# Reference objects with a course-made TRELLIS.2 archive, generated-<object>.tar.gz.
GENERATED_OBJECTS = ("boxed_beverage_001", "hammer_014")
# The TRELLIS.2 model revision those archives were generated with (model_revision in trellis2.json).
GENERATED_MODEL_REVISION = "af44b45f2e35a493886929c6d786e563ec68364d"


def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def _patch_pycolmap_fork():
    # The fork breaks on numpy 2: np.uint64(-1) raises OverflowError.
    import importlib
    import importlib.util
    importlib.invalidate_caches()  # the package was installed by pip in this process
    root = Path(importlib.util.find_spec("pycolmap").submodule_search_locations[0])
    for f in root.rglob("*.py"):
        text = f.read_text()
        if "np.uint64(-1)" in text:
            f.write_text(text.replace("np.uint64(-1)", "np.uint64(2**64-1)"))


if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _pip(RERUN, *PACKAGES[STAGE])
    if STAGE in NO_BUILD_ISOLATION:
        _pip("--no-build-isolation", *NO_BUILD_ISOLATION[STAGE])
    if STAGE in (2, 3):  # stages 2 and 3 read stage1/sparse/0 with the pycolmap fork, never the official wheel
        _patch_pycolmap_fork()
    PROJECT = Path(f"/content/drive/MyDrive/real-to-sim-twin/{OBJECT}")
else:
    if "R2S_PROJECT" not in os.environ:
        raise RuntimeError("Off Colab, set the R2S_PROJECT environment variable to the project folder.")
    PROJECT = Path(os.environ["R2S_PROJECT"])
PROJECT.mkdir(parents=True, exist_ok=True)


@contextlib.contextmanager
def atomic_dir(path):
    """Yield path.tmp; on success it replaces path, on failure path is untouched."""
    path = Path(path)
    tmp, old = path.with_name(path.name + ".tmp"), path.with_name(path.name + ".old")
    shutil.rmtree(tmp, ignore_errors=True)
    if old.exists() and not path.exists():
        old.rename(path)  # a crash between the two renames left the only good copy in .old
    shutil.rmtree(old, ignore_errors=True)
    tmp.mkdir(parents=True)
    try:
        yield tmp
        if path.exists():
            path.rename(old)
        tmp.rename(path)
    except BaseException:
        shutil.rmtree(tmp, ignore_errors=True)
        if old.exists() and not path.exists():
            old.rename(path)
        raise
    shutil.rmtree(old, ignore_errors=True)


def prev_folder(stage):
    """The folder a stage reads: capture/ for stage 1, stage4/ for stage 4b, stage<N-1>/ otherwise.

    Stage 5 reads stage4/ too: stage 4b refines the maps but leaves the mesh to stage 4.
    """
    if stage == 1:
        return "capture"
    if stage == "4b":
        return "stage4"
    return f"stage{stage - 1}"


def on_reference(stage):
    """True if this stage runs on the reference object.

    The reference archives mark each stage folder with an empty .reference file, and a stage
    run on the reference marks the folder it writes. So the answer comes from the stage's own
    input folder (prev_folder(stage)), not from ground_truth/, which
    stays in the project after any fallback. For stage 1 a marked capture/ that also holds files
    of your own (own_capture_files) is not the reference.
    """
    if not (PROJECT / prev_folder(stage) / ".reference").is_file():
        return False
    return not (stage == 1 and own_capture_files(PROJECT / "capture"))


VIDEO_SUFFIXES = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")


def own_capture_files(capture) -> list:
    """Files in capture/ that the reference capture never holds, so they must be yours.

    The reference capture has no board.json, and its only video is video.mp4, extracted at the same
    time as its .reference marker. So board.json, a video under another name (an iPhone's .mov, say)
    or a video changed more than a minute after the marker means that you added your own capture to
    a folder where a reference run left its files. board.json is the signal that does not depend on
    file times; your own capture always needs it for the metric scale.
    """
    capture = Path(capture)
    marker = capture / ".reference"
    since = marker.stat().st_mtime if marker.exists() else float("-inf")
    own = ["board.json"] if (capture / "board.json").exists() else []
    for p in sorted(capture.glob("*")):
        if p.suffix.lower() in VIDEO_SUFFIXES and (p.name != "video.mp4" or p.stat().st_mtime > since + 60):
            own.append(p.name)
    return own


def capture_video(capture):
    """The capture's video: video.mp4, or the one video file of another common phone format."""
    capture = Path(capture)
    if (capture / "video.mp4").exists():
        return capture / "video.mp4"
    videos = [p for p in sorted(capture.glob("*")) if p.suffix.lower() in VIDEO_SUFFIXES]
    if len(videos) != 1:
        raise FileNotFoundError(f"capture/ needs exactly one video (video.mp4, or one .mov, .m4v, .avi, .mkv or "
                                f".webm file); it holds {[p.name for p in videos] or 'none'}")
    return videos[0]


def _extract_reference(archive, names=None, dest=None):
    """Download a reference archive and move its top-level folders (or only `names`) into dest.

    dest defaults to PROJECT. A non-empty folder of yours that a reference folder replaces is
    kept as <name>.yours (an older .yours is replaced); ground_truth/ is replaced without a copy.
    Nothing moves until the archive is fully extracted.
    """
    dest = PROJECT if dest is None else Path(dest)
    base = os.environ.get("R2S_REFERENCE_URL", REFERENCE_URL).rstrip("/")
    tmp = dest / f".{archive}.part"
    req = urllib.request.Request(f"{base}/{archive}", headers={"User-Agent": "real-to-sim-twin"})
    with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    scratch = dest / ".extract"
    shutil.rmtree(scratch, ignore_errors=True)
    try:
        with tarfile.open(tmp) as tar:
            tar.extractall(scratch, filter="data")
        for item in scratch.iterdir():  # move into place only after a complete extraction
            if names is not None and item.name not in names:
                continue
            target = dest / item.name
            if target.is_dir() and any(target.iterdir()) and item.name != "ground_truth":
                yours = dest / f"{item.name}.yours"
                shutil.rmtree(yours, ignore_errors=True)
                target.rename(yours)
                print(f"your {item.name}/ is kept as {yours.name}/ and replaced by the reference {item.name}/.")
            elif target.is_dir():
                shutil.rmtree(target)
            elif target.exists():
                target.unlink()
            item.rename(target)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
        tmp.unlink(missing_ok=True)


def ensure_reference_capture(stage):
    """Stages 5 and 6 on the reference object use the reference capture/ too. True if it fetched it.

    When the stage's input folder is the reference object's (on_reference(stage), which follows
    the .reference markers from any earlier fallback) but capture/ is not (no capture/.reference),
    your measurements in capture/ belong to a different object than the mesh. The reference
    capture/ (physical.json, and the second-light files for stage 6) is then taken from the
    previous stage's archive, and yours is kept as capture.yours/.
    """
    if stage not in (5, 6) or not on_reference(stage) or (PROJECT / "capture" / ".reference").is_file():
        return False
    archive = f"{prev_folder(stage)}.tar.gz"
    print(f"{prev_folder(stage)}/ holds the reference object, so this stage uses the reference capture/ ({archive}).")
    _extract_reference(archive, names={"capture"})
    return True


def ensure_inputs(stage):
    """True if it fell back to the reference object, False if your inputs were present.

    The reference archive for a stage is cumulative: it holds every earlier stage folder that
    later stages read, plus ground_truth/, so a fallback gives one consistent object (see
    _extract_reference for what happens to folders of yours). In stages 5 and 6 it then makes
    sure that a reference object also gets the reference capture/ (ensure_reference_capture).
    """
    prev = prev_folder(stage)
    folder = PROJECT / prev
    fell_back = not (folder.is_dir() and any(folder.iterdir()))
    if fell_back:
        archive = "stage0.tar.gz" if stage == 1 else f"{prev}.tar.gz"
        print(f"{prev}/ is missing or empty: running on the reference object ({archive}).")
        _extract_reference(archive)
    ensure_reference_capture(stage)
    return fell_back


def ensure_generated(obj):
    """Fetch the course's TRELLIS.2 archive for a reference object and return its folder.

    generated-<obj>.tar.gz holds that object's stage folders, ground_truth/ and trellis2/. They
    go to generated/<obj>/, which is replaced whole on each download, so your own stage folders
    are never touched. A failed download leaves the previous generated/<obj>/ as it was. When
    generated/<obj>/trellis2/trellis2.json is already there with GENERATED_MODEL_REVISION, the
    folder is kept and nothing is downloaded.
    """
    if obj not in GENERATED_OBJECTS:
        raise ValueError(f"no generated archive for {obj!r}; the course ships {', '.join(GENERATED_OBJECTS)}")
    out = PROJECT / "generated" / obj
    try:
        revision = json.loads((out / "trellis2" / "trellis2.json").read_text()).get("model_revision")
    except (OSError, ValueError, AttributeError):
        revision = None
    if revision == GENERATED_MODEL_REVISION:
        print(f"generated/{obj}/ is already here (TRELLIS.2 revision {revision[:7]}), so it is not downloaded again.")
        return out
    with atomic_dir(out) as tmp:
        _extract_reference(f"generated-{obj}.tar.gz", dest=tmp)
    return out


import rerun as rr

RRD_PATH = PROJECT / "rrd" / f"stage{STAGE}.rrd"
RRD_PATH.parent.mkdir(parents=True, exist_ok=True)
rr.init("real-to-sim-twin", recording_id=RECORDING_ID)
rr.save(RRD_PATH)
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)


def show_rerun():
    """Show everything logged so far in an inline viewer. Call it last in the stage.

    The viewer replaces the file sink, which leaves stage<N>.rrd complete; the viewer only gets
    data logged after it attaches, so the file is replayed into it. Later log calls reach the
    viewer only, not the file.
    """
    rr.notebook_show()
    rr.log_file_from_path(RRD_PATH)

In this section you keep the stage 4 mesh fixed and adjust its three maps, together with a model of the light around the object, until renders of the mesh match your training photos. Stage 4 painted the maps from Marigold-IID predictions one photo at a time, and nothing in them records where the light came from. A render of the baked mesh therefore has no shading, and on the held-out frames it scores far below the stage 3 splat. Here a renderer that includes a light compares its images with the photos and moves the maps and the light by gradient descent. This is inverse rendering.

The fit explains your photos better than the bake does, but it does not tell you what the object is made of. Albedo brightness and light intensity can trade against each other without changing the rendered diffuse color at all, and under one uncontrolled light the photos say little about metallic. You score the bake, fit the object three ways and measure both effects on your object. Stages 5 and 6 keep the stage 4 maps for the physics and the exported asset, and stage 6 scores the refined maps beside them.

The mesh is rasterized and its textures are sampled with nvdiffrast 0.4.0 (Laine et al., "Modular Primitives for High-Performance Differentiable Rendering", SIGGRAPH Asia 2020), released under the NVIDIA Source Code License, which allows non-commercial use only. The shading is course code. It follows the approach of nvdiffrec (Munkberg et al., "Extracting Triangular 3D Models, Materials, and Lighting From Images", CVPR 2022) and uses none of its code.

Runtime budget on a Colab T4: about 2 minutes for the four runs once the session is set up, with about 1.3 GB of GPU memory in use. The first run in a new session takes about 6 minutes, of which about 4 minutes is one-time setup.

In [ ]:
import json
import time

import cv2
import numpy as np

T0 = time.time()
FALLBACK = ensure_inputs(STAGE)
STAGE1, STAGE2, STAGE4 = PROJECT / "stage1", PROJECT / "stage2", PROJECT / "stage4"
REFERENCE = on_reference(STAGE)  # the reference stage4/ carries a .reference marker
frames = sorted((STAGE1 / "frames").glob("*.jpg"))
heldout = set((STAGE1 / "heldout.txt").read_text().split())
drifting = set(json.loads((STAGE2 / "drift.json").read_text())["flagged"])
train = [p for p in frames if p.name not in heldout and p.name not in drifting]
test = [p for p in frames if p.name in heldout]
masks = {p.name: cv2.imread(str(STAGE2 / "masks" / (p.stem + ".png")), cv2.IMREAD_GRAYSCALE) > 127 for p in train + test}
SCAN_OBJECT = json.loads((PROJECT / "ground_truth" / "scan.json").read_text())["object"] if REFERENCE else None
print("object:", f"reference ({SCAN_OBJECT})" if REFERENCE else OBJECT)
print(f"{len(frames)} frames: {len(train)} for training, {len(test)} held out for scoring")

In [ ]:
# Stage 6's loaders and score functions, copied unchanged. This section renders with its own renderer, so
# the same maps can score up to about 0.6 dB differently here and in stage 6.
import cv2
import numpy as np
from pathlib import Path


def opencv_distortion(model: str, params) -> np.ndarray:
    """OpenCV's (k1, k2, p1, p2) for a COLMAP camera model and its parameter list."""
    params = list(params)
    if model in ("PINHOLE", "SIMPLE_PINHOLE"):
        return np.zeros(4)
    if model == "SIMPLE_RADIAL":
        return np.array([params[3], 0.0, 0.0, 0.0])
    if model == "RADIAL":
        return np.array([params[3], params[4], 0.0, 0.0])
    if model == "OPENCV":
        return np.array(params[4:8], dtype=float)
    raise ValueError(f"camera model {model} is not supported here")


def load_views(stage1: Path, frames, masks: dict, data_factor: int = 2) -> list:
    """Undistorted photos and masks at 1/data_factor resolution with their cameras ("K", and
    "viewmat", world to camera in OpenCV axes), as in stage 4."""
    import pycolmap

    model = pycolmap.Reconstruction(str(Path(stage1) / "sparse" / "0"))
    by_name = {im.name: im for im in model.images.values()}
    views, skipped = [], []
    for p in frames:
        im = by_name.get(Path(p).name)
        if im is None:
            skipped.append(Path(p).name)
            continue
        cam = model.cameras[im.camera_id]
        K = cam.calibration_matrix().copy()
        K[:2] /= data_factor
        dist = opencv_distortion(cam.model.name, cam.params)
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        size = (img.shape[1] // data_factor, img.shape[0] // data_factor)
        img = cv2.undistort(cv2.resize(img, size, interpolation=cv2.INTER_AREA), K, dist)
        m = cv2.resize(masks[Path(p).name].astype(np.float32), size, interpolation=cv2.INTER_AREA)
        viewmat = np.eye(4)
        viewmat[:3] = im.cam_from_world().matrix()
        views.append({"name": Path(p).name, "K": K, "viewmat": viewmat, "image": img,
                      "mask": cv2.undistort(m, K, dist) > 0.5})
    if skipped:  # write_stage1 accepts a model with 80% of the frames registered
        print(f"{len(skipped)} of {len(frames)} frames have no pose in stage1/sparse/0 (stage 1 did not register "
              f"them) and are left out: {', '.join(skipped)}")
    return views


def srgb_to_linear(c):
    c = np.clip(c, 0, 1)
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)


def linear_to_srgb(c):
    c = np.clip(c, 0, 1)
    return np.where(c <= 0.0031308, 12.92 * c, 1.055 * c ** (1 / 2.4) - 0.055)


def masked_psnr(image: np.ndarray, photo: np.ndarray, mask: np.ndarray) -> float:
    """PSNR (dB) over the mask's pixels, for images in [0, 1]."""
    mse = float(((image - photo)[mask] ** 2).mean())
    return float(10 * np.log10(1.0 / max(mse, 1e-12)))


def scoreable_views(views: list, flagged) -> tuple[list, list]:
    """Split views into those the masked scores can use and (name, reason) for the rest: a view whose
    mask is empty has no pixels to score, and a mask the stage 2 drift check flagged is not trusted."""
    kept, left_out = [], []
    for v in views:
        if not v["mask"].any():
            left_out.append((v["name"], "empty mask"))
        elif v["name"] in flagged:
            left_out.append((v["name"], "mask flagged by the stage 2 drift check"))
        else:
            kept.append(v)
    return kept, left_out


def masked_lpips(metric, image: np.ndarray, photo: np.ndarray, mask: np.ndarray, margin: int = 16) -> float:
    """LPIPS between image and photo with everything outside the mask set to white in both, on the
    mask's bounding box plus a margin, so that the score is about the object."""
    ys, xs = np.nonzero(mask)
    y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin + 1, mask.shape[0])
    x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin + 1, mask.shape[1])
    crop = lambda a: np.where(mask[..., None], a, 1.0)[y0:y1, x0:x1]
    import torch

    dev = next(metric.parameters()).device
    t = lambda a: torch.tensor(crop(a), dtype=torch.float32, device=dev).permute(2, 0, 1)[None]
    with torch.no_grad():
        return float(metric(t(image), t(photo), normalize=True))


def gain_match(image: np.ndarray, photo: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, float]:
    """Scale the image's linear intensity by the one gain that best fits the photo on the mask (least
    squares), as an exposure correction; returns the corrected image and the gain."""
    a, b = srgb_to_linear(image)[mask], srgb_to_linear(photo)[mask]
    gain = float((a * b).sum() / max((a * a).sum(), 1e-12))
    return linear_to_srgb(srgb_to_linear(image) * gain), gain


def load_mesh_uv(path: Path) -> dict:
    """Read stage4/mesh_uv.obj: "vertices" (V x 3, meters), "faces" (F x 3), "normals" (V x 3) and
    "uvs" (V x 2, origin at the top-left of the texture, v down; the OBJ stores v up)."""
    v, vt, vn, f = [], [], [], []
    for line in Path(path).read_text().splitlines():
        if line.startswith("v "):
            v.append(line.split()[1:4])
        elif line.startswith("vt "):
            vt.append(line.split()[1:3])
        elif line.startswith("vn "):
            vn.append(line.split()[1:4])
        elif line.startswith("f "):
            f.append([int(w.split("/")[0]) - 1 for w in line.split()[1:4]])
    uvs = np.asarray(vt, dtype=float)
    if len(uvs) != len(v):  # stage 4 writes f a/a/a: vertex i has texture coordinate i
        raise ValueError(f"{path} has {len(v)} vertices and {len(uvs)} texture coordinates; stage 6 reads the "
                         "stage 4 format, one texture coordinate per vertex")
    return {"vertices": np.asarray(v, dtype=float), "faces": np.asarray(f, dtype=np.int64),
            "normals": np.asarray(vn, dtype=float) if len(vn) == len(v) else None,
            "uvs": np.c_[uvs[:, 0], 1 - uvs[:, 1]]}

## Light leaving a surface

The light that leaves a surface point toward the camera, in the direction $\mathbf v$, is given by the rendering equation without emission:

$$
L_o(\mathbf v) = \int_{\Omega} f(\mathbf l, \mathbf v) \, L_i(\mathbf l) \, (\mathbf n \cdot \mathbf l)^+ \, d\mathbf l ,
$$

where $\mathbf n$ is the surface normal, $L_i(\mathbf l)$ the radiance arriving from the direction $\mathbf l$, $f$ the BRDF and $(x)^+ = \max(x, 0)$. Here the light is a *distant environment*: $L_i$ depends on the direction only, not on the point, so every point of the object sees the same sky. Nothing in the model lets one part of the object block light from another. The light becomes a single function on the sphere, stored as a 32 by 64 latitude-longitude map of linear RGB radiance.

The BRDF is the metallic workflow from stage 4. A point with linear base color $k_d$, roughness $r$ and metallic $m$ reflects a diffuse part with color $k_d (1 - m)$ and a specular part whose reflectance at normal incidence is

$$
F_0 = 0.04 \, (1 - m) + k_d \, m .
$$

A dielectric reflects 4% at normal incidence whatever its color. A metal has no diffuse part and tints its reflection with $k_d$. The diffuse part is Lambertian, $f_d = k_d (1 - m) / \pi$, so it needs only the irradiance

$$
E(\mathbf n) = \frac{1}{\pi} \int_{\Omega} L_i(\mathbf l) \, (\mathbf n \cdot \mathbf l)^+ \, d\mathbf l ,
$$

which depends on the normal alone. Over the 2,048 texels of the light map the integral is a weighted sum, computed exactly for every texel direction, and a uniform light of radiance $L$ gives $E = L$. The diffuse term of a pixel is $k_d (1 - m) \, E(\mathbf n)$.

## The split-sum approximation

The specular part is a Cook-Torrance microfacet lobe with the GGX distribution $D$, a Fresnel term $F$ and a shadowing term $G$:

$$
f_s(\mathbf l, \mathbf v) = \frac{D(\mathbf h) \, F(\mathbf v \cdot \mathbf h) \, G(\mathbf l, \mathbf v)}{4 \, (\mathbf n \cdot \mathbf l)(\mathbf n \cdot \mathbf v)}, \qquad
D(\mathbf h) = \frac{\alpha^2}{\pi \left( (\mathbf n \cdot \mathbf h)^2 (\alpha^2 - 1) + 1 \right)^2}, \qquad \alpha = r^2 ,
$$

with $\mathbf h$ the half vector between $\mathbf l$ and $\mathbf v$. Integrating $f_s \, L_i \, (\mathbf n \cdot \mathbf l)^+$ over the sphere at every pixel and every step costs too much. The split-sum approximation (Karis, "Real Shading in Unreal Engine 4", SIGGRAPH 2013 course notes) cuts the integral into two factors that can be computed ahead of time:

$$
\int_\Omega f_s \, L_i \, (\mathbf n \cdot \mathbf l)^+ \, d\mathbf l \;\approx\;
\frac{\int_\Omega D \, L_i \, (\mathbf n \cdot \mathbf l)^+ \, d\mathbf l}{\int_\Omega D \, (\mathbf n \cdot \mathbf l)^+ \, d\mathbf l}
\;\cdot\; \int_\Omega f_s \, (\mathbf n \cdot \mathbf l)^+ \, d\mathbf l .
$$

The first factor is the light averaged under the GGX lobe. It still depends on $\mathbf n$ and $\mathbf v$, and the approximation removes that dependence by assuming $\mathbf n = \mathbf v = \mathbf r$, where $\mathbf r$ is the mirror reflection of $\mathbf v$ about $\mathbf n$. The lobe is taken to point along the reflection direction with the shape it has when you look at the surface head on. The prefiltered light is then a function of $\mathbf r$ and $r$ alone. Here it is stored as seven maps, one per roughness from 0.08 to 1.0, and each pixel interpolates between the two levels around its roughness.

The second factor is the BRDF integrated against a white light. It depends on $r$, on $\mathbf n \cdot \mathbf v$ and on $F_0$, and since Schlick's Fresnel term is linear in $F_0$ it splits into $F_0 A + B$. The code uses the polynomial fit of $A$ and $B$ that Unreal Engine runs on mobile hardware (Karis, "Physically Based Shading on Mobile", 2014). It is off by up to about 0.15 in $A$ and costs a few multiplications. The specular term of a pixel is the prefiltered light along $\mathbf r$ times $F_0 A + B$.

The assumption fails where the real lobe differs most from the head-on one. Seen at a grazing angle, a real reflection lobe stretches and leans toward the surface, so highlights near the silhouette come out too round and slightly out of place.

In [ ]:
import math

import torch


def srgb_to_linear_t(c):
    """sRGB values in [0, 1] to linear intensity (torch; stage 6's srgb_to_linear works on numpy)."""
    c = c.clamp(0, 1)
    return torch.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055).clamp_min(1e-8) ** 2.4)


def linear_to_srgb_t(c):
    """Linear intensity to sRGB in [0, 1]; values above 1 saturate, as in an 8-bit photo."""
    c = c.clamp(0, 1)
    return torch.where(c <= 0.0031308, 12.92 * c, 1.055 * c.clamp_min(1e-8) ** (1 / 2.4) - 0.055)


def latlong_dirs(h: int, w: int, device):
    """Unit directions (h * w, 3) and solid angles (h * w,) of the texel centers of an h x w
    latitude-longitude map in the stage 1 world frame: row 0 at +Z (the pole above the object),
    columns running in longitude from -pi to pi."""
    theta = (torch.arange(h, device=device) + 0.5) / h * math.pi
    phi = (torch.arange(w, device=device) + 0.5) / w * 2 * math.pi - math.pi
    th, ph = torch.meshgrid(theta, phi, indexing="ij")
    d = torch.stack([torch.sin(th) * torch.cos(ph), torch.sin(th) * torch.sin(ph), torch.cos(th)], -1)
    domega = torch.sin(th) * (math.pi / h) * (2 * math.pi / w)
    return d.reshape(-1, 3), domega.reshape(-1)


ROUGH_LEVELS = (0.08, 0.15, 0.25, 0.4, 0.6, 0.8, 1.0)  # roughness of each prefiltered specular map


class EnvLight(torch.nn.Module):
    """A learnable distant light: an h x w latitude-longitude map of linear RGB radiance, fixed in
    the world frame and stored as log radiance so that it stays positive.

    prefiltered() returns the two maps the shading reads. The diffuse map holds the irradiance
    (1/pi) * integral L(l) max(n . l, 0) dl for each normal direction n, by quadrature over the
    texels, so a uniform light of radiance L gives L and a Lambert surface of albedo a returns a * L.
    The specular maps hold, for each roughness in ROUGH_LEVELS, the GGX-weighted average of L around
    a reflection direction r with n = v = r (the split-sum assumption): weights D(n . h) max(n . l, 0) dl,
    normalized to sum to one.
    """

    def __init__(self, h: int = 32, w: int = 64, init: float = 0.6, device="cuda"):
        super().__init__()
        self.h, self.w = h, w
        self.log_L = torch.nn.Parameter(torch.full((h * w, 3), math.log(init), device=device))
        d, dom = latlong_dirs(h, w, device)
        cos = d @ d.T  # (out, in): cosine between an output direction and a light texel
        self.register_buffer("W_diff", cos.clamp_min(0) * dom[None] / math.pi)
        ndh = ((1 + cos) / 2).clamp_min(0).sqrt()  # n . h for the half vector of n and l when v = n
        mats = []
        for r in ROUGH_LEVELS:
            a2 = (r * r) ** 2
            D = a2 / (math.pi * ((ndh * ndh) * (a2 - 1) + 1) ** 2)
            wgt = D * cos.clamp_min(0) * dom[None]
            mats.append(wgt / wgt.sum(1, keepdim=True).clamp_min(1e-12))
        self.register_buffer("W_spec", torch.stack(mats))  # (levels, out, in)

    def radiance(self):
        """Linear radiance, (h * w, 3)."""
        return self.log_L.exp()

    def prefiltered(self):
        """(diffuse map (1, h, w, 3), specular maps (levels, h, w, 3))."""
        L = self.radiance()
        diff = (self.W_diff @ L).reshape(1, self.h, self.w, 3)
        spec = torch.matmul(self.W_spec, L).reshape(len(ROUGH_LEVELS), self.h, self.w, 3)
        return diff, spec

    def lookup(self, maps, d):
        """Bilinear lookup of maps (K, h, w, 3) along unit directions d (P, 3); returns (K, P, 3).
        Longitude wraps around; latitude clamps at the poles."""
        K = maps.shape[0]
        theta = torch.acos(d[:, 2].clamp(-1 + 1e-6, 1 - 1e-6))
        phi = torch.atan2(d[:, 1], d[:, 0])
        x = (phi + math.pi) / (2 * math.pi) * self.w + 1.0  # + 1: one wrapped column on each side
        y = theta / math.pi * self.h
        padded = torch.cat([maps[:, :, -1:], maps, maps[:, :, :1]], 2).permute(0, 3, 1, 2)  # K, 3, h, w + 2
        grid = torch.stack([2 * x / (self.w + 2) - 1, 2 * y / self.h - 1], -1)
        out = torch.nn.functional.grid_sample(padded, grid[None, None].expand(K, 1, -1, 2), mode="bilinear",
                                              padding_mode="border", align_corners=False)
        return out[:, :, 0].permute(0, 2, 1)

    def white_reg(self):
        """Mean departure of the light from gray, relative to its mean: penalizes a colored light, so
        that the color of the object goes into the albedo."""
        L = self.radiance()
        return (L - L.mean(-1, keepdim=True)).abs().mean() / L.mean().detach().clamp_min(1e-6)


def env_brdf_approx(rough, nov):
    """Karis' analytic fit of the split-sum DFG term (Unreal Engine's EnvBRDFApprox, from "Physically Based
    Shading on Mobile", 2014, after Lazarov 2013): (A, B) with the integral of the BRDF approximately
    F0 * A + B. The fit is coarse, off by up to about 0.15 in A against a numerical integral."""
    c0 = torch.tensor([-1.0, -0.0275, -0.572, 0.022], device=rough.device)
    c1 = torch.tensor([1.0, 0.0425, 1.04, -0.04], device=rough.device)
    r = rough * c0 + c1
    a004 = torch.minimum(r[:, 0:1] ** 2, torch.exp2(-9.28 * nov)) * r[:, 0:1] + r[:, 1:2]
    return -1.04 * a004 + r[:, 2:3], 1.04 * a004 + r[:, 3:4]

In [ ]:
import torch


def shade(kd_srgb, rough, metal, n, v, env, pref, parts: bool = False):
    """Linear radiance (P, 3) leaving P surface points toward the camera, or (diffuse, specular)
    when parts is True.

    kd_srgb (P, 3) is the sRGB base color, rough and metal (P, 1), n and v (P, 3) the unit normal
    and the unit direction to the camera, env an EnvLight and pref = (diffuse map, specular maps),
    the output of env.prefiltered().

    1. kd = srgb_to_linear_t(kd_srgb): the shading works in linear intensity.
    2. n . v per point, clamped below at 1e-4, and the reflection direction
       r = normalize(2 (n . v) n - v).
    3. The irradiance E = env.lookup(diffuse map, n)[0], shape (P, 3), and the prefiltered light
       along r at every roughness level, env.lookup(specular maps, r), shape (len(ROUGH_LEVELS), P, 3).
    4. Clamp the roughness to [ROUGH_LEVELS[0], 1] and interpolate the prefiltered light linearly
       between the two levels around it (torch.searchsorted finds the index of the level).
    5. (A, B) = env_brdf_approx(clamped roughness, n . v) and F0 = 0.04 (1 - m) + kd m.
    6. diffuse = kd (1 - m) E and specular = (interpolated light) (F0 A + B).
    Return (diffuse, specular) if parts, else diffuse + specular.
    """
    raise NotImplementedError

## A light fixed in the world

You walked around the object while it stood still on the table, so the light in the room was the same for every frame. The environment map is therefore one map in the stage 1 world frame, shared by every view, with row 0 straight up ($+z$) and the columns running in longitude from $-\pi$ to $\pi$.

The diffuse term depends only on the normal, so a point keeps the same diffuse color in every frame. The specular term depends on the reflection direction $\mathbf r$, which turns with the camera, so highlights slide over the surface from frame to frame. These moving highlights are the only evidence your photos hold about roughness and metallic. A rough surface spreads a highlight wide and dim, a smooth one keeps it small and bright, and a metal tints it with its own color. A part of the object that never catches a highlight from any camera position gives the fit nothing to work with, and its roughness and metallic stay near the values the bake gave them.

In [ ]:
# Rasterization, attribute interpolation and mipmapped texture sampling use nvdiffrast 0.4.0, whose
# license is the NVIDIA Source Code License (non-commercial use only). The shading is shade() above.
import numpy as np
import nvdiffrast.torch as dr
import torch


class Scene:
    """The fixed stage 4 mesh (mesh_uv as load_mesh_uv returns it, with vertex normals) on the GPU."""

    def __init__(self, mesh_uv: dict, device="cuda"):
        t = lambda a, dtype=torch.float32: torch.tensor(np.ascontiguousarray(a), dtype=dtype, device=device)
        self.v, self.f = t(mesh_uv["vertices"]), t(mesh_uv["faces"], torch.int32)
        self.uv = t(mesh_uv["uvs"])
        normals = mesh_uv.get("normals")
        if normals is None:
            import trimesh
            normals = trimesh.Trimesh(mesh_uv["vertices"], mesh_uv["faces"], process=False).vertex_normals
        self.n = torch.nn.functional.normalize(t(normals), dim=-1)
        self.ctx = dr.RasterizeCudaContext(device=device)

    def clip(self, K, M, w, h, near=0.01, far=10.0):
        """Clip-space vertices of a pinhole camera K and world-to-camera M (OpenCV axes), as stage 6."""
        cam = self.v @ M[:3, :3].T + M[:3, 3]
        X, Y, Z = cam.unbind(-1)
        return torch.stack([(2 * K[0, 0] * X + (2 * K[0, 2] - w) * Z) / w,
                            (2 * K[1, 1] * Y + (2 * K[1, 2] - h) * Z) / h,
                            (far + near) / (far - near) * Z - 2 * far * near / (far - near), Z], -1)

    def gbuffer(self, Ks, Ms, w, h):
        """G-buffer of B views of one size: Ks (B, 3, 3), Ms (B, 4, 4) torch tensors on the device.
        Returns "covered" (B, h, w), "uv" and "uv_da" (texture coordinates and their screen
        derivatives, for mipmapping), "pos" and "normal" (B, h, w, 3), row 0 at the image top."""
        clip = torch.stack([self.clip(K, M, w, h) for K, M in zip(Ks, Ms)]).contiguous()
        rast, rast_db = dr.rasterize(self.ctx, clip, self.f, (h, w))
        uv, uv_da = dr.interpolate(self.uv[None], rast, self.f, rast_db=rast_db, diff_attrs="all")
        pos, _ = dr.interpolate(self.v[None], rast, self.f)
        nrm, _ = dr.interpolate(self.n[None], rast, self.f)
        return {"covered": rast[..., 3] > 0, "uv": uv, "uv_da": uv_da, "pos": pos,
                "normal": torch.nn.functional.normalize(nrm, dim=-1)}

    def chart_mask(self, size: int):
        """(size, size) bool: texels whose center lies inside a UV chart (the UV layout rasterized)."""
        clip = torch.cat([self.uv * 2 - 1, torch.zeros_like(self.uv[:, :1]), torch.ones_like(self.uv[:, :1])], -1)
        rast, _ = dr.rasterize(self.ctx, clip[None].contiguous(), self.f, (size, size))
        return rast[0, ..., 3] > 0  # row 0 is v = 0, as in texture memory


class Material(torch.nn.Module):
    """The three maps in one learnable (1, size, size, 5) tensor: sRGB base color, roughness,
    metallic, resized from the given arrays (values in [0, 1]); init keeps the starting values."""

    def __init__(self, base_color, roughness, metallic, size: int, device="cuda"):
        super().__init__()
        import cv2

        rs = lambda a: cv2.resize(np.asarray(a, np.float32), (size, size), interpolation=cv2.INTER_AREA)
        tex = np.concatenate([rs(base_color), rs(roughness)[..., None], rs(metallic)[..., None]], -1)
        self.tex = torch.nn.Parameter(torch.tensor(tex, device=device)[None])
        self.register_buffer("init", self.tex.detach().clone())

    def clamp_(self):
        """Keep the maps in range after an optimizer step; roughness stays in [0.08, 1]."""
        with torch.no_grad():
            self.tex[..., :3].clamp_(0.0, 1.0)
            self.tex[..., 3].clamp_(0.08, 1.0)
            self.tex[..., 4].clamp_(0.0, 1.0)

    def sample(self, g):
        return dr.texture(self.tex, g["uv"], g["uv_da"], filter_mode="linear-mipmap-linear", boundary_mode="clamp")


def render_batch(scene, mat, env, Ks, Ms, centers, w, h, pref=None):
    """Relit sRGB images (B, h, w, 3) on white, the coverage (B, h, w) and the sampled maps
    (B, h, w, 5). centers (B, 3) are the camera centers in the world; pref, if given, is
    env.prefiltered() computed once for several calls."""
    g = scene.gbuffer(Ks, Ms, w, h)
    m = mat.sample(g)
    cov = g["covered"]
    idx = cov.nonzero(as_tuple=True)
    pos, n = g["pos"][idx], g["normal"][idx]
    v = torch.nn.functional.normalize(centers[idx[0]] - pos, dim=-1)
    if pref is None:
        pref = env.prefiltered()
    mm = m[idx]
    lin = shade(mm[:, :3], mm[:, 3:4], mm[:, 4:5], n, v, env, pref)
    img = torch.ones(cov.shape + (3,), device=cov.device)
    img = img.index_put(idx, linear_to_srgb_t(lin))
    return img, cov, m

## Fitting the maps and the light

Every step renders four random training views with the current maps and light, and compares them with the photos:

$$
\mathcal L = \frac{1}{|M|} \sum_{p \in M} \lVert \hat I_p - I_p \rVert_1
+ \sum_{c} w^{\mathrm{tv}}_c \, \mathrm{TV}(T_c)
+ \sum_{c} w^{\mathrm{pull}}_c \, \overline{\lvert T_c - T^{0}_c \rvert}
+ w_{\mathrm{white}} \, W(L) .
$$

The first term is the L1 difference in sRGB between the render $\hat I$ and the photo $I$ over the pixels $M$ that lie inside both the rendered silhouette and the stage 2 mask, eroded by 2 pixels so that its edge does not count. The photos are 8-bit sRGB video frames, so the comparison happens in sRGB. $\mathrm{TV}(T_c)$ is the total variation of map $c$ (albedo, roughness, metallic) between neighboring texels of the same UV chart, with weights 0.5, 1.0 and 1.0. Two texels on either side of a seam sit next to each other in the texture but far apart on the surface, so pairs across a chart boundary are left out. The pull is the mean absolute difference between each map and its starting value $T^0_c$, the stage 4 bake, over the chart texels, with weights 0.02, 0.01 and 0.01. $W(L)$ measures how far the light departs from gray, relative to its mean, with weight 0.01. Without it a colored light could explain the color of the object, and this term pushes the color into the albedo.

Adam moves the 1024 by 1024 maps at a learning rate of 0.01 and the logarithm of the light's radiance at 0.03, both falling tenfold over 1,000 steps. The training frames are at half resolution, 960 by 540 pixels, as in stage 6. After each step the maps are clamped to $[0, 1]$ with roughness kept at 0.08 or above. Below that the GGX lobe is narrower than one texel of the 32 by 64 light, and the specular maps cannot represent it.

The pull is there because of an ambiguity the photos cannot resolve. Scale the linear albedo by $c$ and the light by $1/c$. The irradiance becomes $E / c$, and the diffuse term $c \, k_d (1 - m) \cdot E / c$ is unchanged at every pixel of every view. Only the specular term notices. A dielectric's $F_0 = 0.04$ does not follow $k_d$, so its highlight dims by $1/c$, and on a mostly matte object that term is a few percent of the pixel. The photos therefore fix the scale only weakly, and at these weights the pull is a weak brake too. On the reference object, the `refined` run below still darkens the albedo mean from (0.713, 0.858, 0.562) to (0.641, 0.778, 0.500) in sRGB, about 0.79 of the bake in linear terms, and the `no_pull` run darkens it by only a further factor of 0.948. The scale the fit ends at comes mostly from where it starts, the bake's maps and a light of radiance 0.6, and from how far 1,000 steps move them. The cell below shows the trade on a single point.

In [ ]:
# The scale trade in numbers: one point with n = v = +z, roughness 0.5 and metallic 0 under a uniform light.
z = torch.tensor([[0.0, 0.0, 1.0]])
for kd_linear, radiance in ((0.4, 0.8), (0.8, 0.4)):
    light = EnvLight(8, 16, init=radiance, device="cpu")
    with torch.no_grad():
        diffuse, specular = shade(linear_to_srgb_t(torch.full((1, 3), kd_linear)), torch.full((1, 1), 0.5),
                                  torch.zeros(1, 1), z, z, light, light.prefiltered(), parts=True)
    print(f"linear albedo {kd_linear}, light {radiance}: diffuse {diffuse[0, 0]:.4f}, specular {specular[0, 0]:.4f}")

In [ ]:
import torch


def tv_loss(tex, chart):
    """Total variation of the maps tex (1, S, S, 5): the mean absolute difference between
    horizontally and vertically adjacent texels that both lie inside a chart (chart (S, S) bool),
    so a seam between two charts costs nothing. Returns (albedo, roughness, metallic)."""
    t = tex[0]
    dx = (t[:, 1:] - t[:, :-1]).abs()
    dy = (t[1:] - t[:-1]).abs()
    mx = (chart[:, 1:] & chart[:, :-1]).to(t.dtype)[..., None]
    my = (chart[1:] & chart[:-1]).to(t.dtype)[..., None]
    per = ((dx * mx).sum((0, 1)) + (dy * my).sum((0, 1))) / (mx.sum() + my.sum()).clamp_min(1)
    return per[:3].mean(), per[3], per[4]

In [ ]:
import torch


def refine_loss(render, photo, mask, tex, init_tex, chart, env, w_tv=(0.5, 1.0, 1.0), w_pull=(0.02, 0.01, 0.01),
                w_white=0.01) -> torch.Tensor:
    """The loss of one step: L1 in sRGB between render and photo, plus three regularizers.

    render and photo are (B, h, w, 3) in [0, 1] and mask (B, h, w) bool. tex and init_tex are the
    maps now and at the start, (1, S, S, 5): sRGB base color, roughness, metallic. chart is
    (S, S) bool and env the EnvLight.

    1. l1: the mean of |render - photo| over the masked pixels and the three channels. With an
       empty mask use render.sum() * 0, so that the loss stays connected to the graph.
    2. (tv_albedo, tv_roughness, tv_metallic) = tv_loss(tex, chart).
    3. The pull: for each of the five channels, the mean of |tex - init_tex| over the chart
       texels. pull_albedo is the mean of the first three; pull_roughness and pull_metallic are
       the other two.
    4. Return l1 + w_tv[0] tv_albedo + w_tv[1] tv_roughness + w_tv[2] tv_metallic
       + w_pull[0] pull_albedo + w_pull[1] pull_roughness + w_pull[2] pull_metallic
       + w_white env.white_reg().
    """
    raise NotImplementedError

In [ ]:
import math
import time

import numpy as np
import torch


def view_tensors(views: list, device):
    """Cameras of views (as load_views returns them) as tensors: K (B, 3, 3), world-to-camera
    (B, 4, 4) and the camera centers in the world (B, 3)."""
    Ks = torch.tensor(np.stack([v["K"] for v in views]), dtype=torch.float32, device=device)
    Ms = torch.tensor(np.stack([v["viewmat"] for v in views]), dtype=torch.float32, device=device)
    C = -torch.einsum("bji,bj->bi", Ms[:, :3, :3], Ms[:, :3, 3])
    return Ks, Ms, C


def steps_to_converge(history: list, tol: float = 0.1):
    """The first evaluation step whose held-out PSNR is within tol dB of the final one."""
    final = history[-1]["psnr"]
    return next(h["step"] for h in history if h["psnr"] >= final - tol)


def chart_means(maps: dict, chart) -> dict:
    """Means of the maps over the chart texels: albedo (sRGB, three channels), roughness, metallic."""
    chart = np.asarray(chart, bool)
    return {"albedo_mean_srgb": [float(c) for c in np.asarray(maps["base_color"])[chart].mean(0)],
            "roughness_mean": float(np.asarray(maps["roughness"])[chart].mean()),
            "metallic_mean": float(np.asarray(maps["metallic"])[chart].mean())}


def relit_psnr(scene, mat, env, views: list) -> float:
    """Mean masked PSNR of the relit renders of views against their photos."""
    scores = []
    with torch.no_grad():
        pref = env.prefiltered()
        for v in views:
            h, w = v["image"].shape[:2]
            img, _, _ = render_batch(scene, mat, env, *view_tensors([v], scene.v.device), w, h, pref)
            scores.append(masked_psnr(img[0].cpu().numpy(), v["image"] / 255.0, v["mask"]))
    return float(np.mean(scores))


def refine(scene, train_views, test_views, maps0: dict, *, size=1024, steps=1000, batch=4, env_hw=(32, 64),
           lr_tex=0.01, lr_env=0.03, w_tv=(0.5, 1.0, 1.0), w_pull=(0.02, 0.01, 0.01), w_white=0.01,
           light_only=False, eval_every=100, seed=0) -> dict:
    """Fit the maps and a distant light to the training photos by differentiable rendering.

    maps0 holds the starting maps ("base_color" (S, S, 3) sRGB, "roughness" and "metallic" (S, S),
    in [0, 1]), resized to size. Each step renders batch random training views, takes refine_loss
    against the photos over their masks eroded by 2 px, and moves the maps (unless light_only) and
    the light with Adam; both learning rates fall tenfold over the run. Every eval_every steps the
    held-out relit PSNR goes into the history. seconds counts training time only.
    """
    import cv2

    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    dev = scene.v.device
    mat = Material(maps0["base_color"], maps0["roughness"], maps0["metallic"], size, device=dev)
    chart = scene.chart_mask(size)
    env = EnvLight(*env_hw, init=0.6, device=dev)
    Ks, Ms, C = view_tensors(train_views, dev)
    photos = torch.tensor(np.stack([v["image"] for v in train_views]), device=dev)  # uint8
    kernel = np.ones((5, 5), np.uint8)
    masks = torch.tensor(np.stack([cv2.erode(v["mask"].astype(np.uint8), kernel) > 0 for v in train_views]), device=dev)
    h, w = train_views[0]["image"].shape[:2]
    groups = [{"params": [env.log_L], "lr": lr_env}] + ([] if light_only else [{"params": [mat.tex], "lr": lr_tex}])
    opt = torch.optim.Adam(groups, betas=(0.9, 0.99))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda it: 10 ** (-it / max(steps, 1)))
    sync = torch.cuda.synchronize if mat.tex.is_cuda else (lambda: None)
    history, seconds, seconds_at = [], 0.0, {}
    for step in range(1, steps + 1):
        sync()
        t0 = time.time()
        b = torch.tensor(rng.choice(len(train_views), size=batch, replace=False), device=dev)
        img, cov, _ = render_batch(scene, mat, env, Ks[b], Ms[b], C[b], w, h)
        loss = refine_loss(img, photos[b].float() / 255.0, cov & masks[b], mat.tex, mat.init, chart, env,
                           w_tv, w_pull, w_white)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
        mat.clamp_()
        sync()
        seconds += time.time() - t0
        if step % eval_every == 0 or step == steps:
            history.append({"step": step, "psnr": relit_psnr(scene, mat, env, test_views)})
            seconds_at[step] = seconds
    tex = mat.tex.detach()[0].cpu().numpy().astype(np.float32)
    maps = {"base_color": tex[..., :3].copy(), "roughness": tex[..., 3].copy(), "metallic": tex[..., 4].copy()}
    radiance = env.radiance().detach().reshape(env_hw[0], env_hw[1], 3).cpu().numpy().astype(np.float32)
    converged = steps_to_converge(history)
    return {"maps": maps, "env": radiance, "history": history, "steps_to_converge": converged, "seconds": seconds,
            "seconds_to_converge": seconds_at[converged], "light_mean": float(radiance.mean()),
            "chart": chart.cpu().numpy(), **chart_means(maps, chart.cpu().numpy())}

In [ ]:
import math

import numpy as np
import torch


def evaluate_refined(scene, maps: dict, env, test_views: list, lpips_metric, min_views: int = 3,
                     keep_images: bool = False) -> dict:
    """Held-out scores of the maps, rendered at each view's size on white, as stage 6 scores the
    mesh: masked PSNR and masked LPIPS against the photo, averaged over the views.

    "relit" scores the shaded render under env, the (h, w, 3) linear radiance that refine returns,
    or is None when env is None (the stage 4 bake has no light). "base_color_unlit" scores the base
    color alone and "base_color_matched" the same after gain_match. "views" holds the per-view
    scores; with keep_images, "images" holds the photo, mask and renders of each view.
    test_views must already have passed scoreable_views; fewer than min_views is refused.
    """
    if len(test_views) < min_views:
        raise ValueError(f"only {len(test_views)} scoreable held-out frame(s), at least {min_views} are needed to "
                         "score the maps: check the stage 2 masks of the held-out frames")
    dev = scene.v.device
    size = np.asarray(maps["roughness"]).shape[0]
    mat = Material(maps["base_color"], maps["roughness"], maps["metallic"], size, device=dev)
    light = EnvLight(*(np.asarray(env).shape[:2] if env is not None else (32, 64)), device=dev)
    if env is not None:
        with torch.no_grad():
            light.log_L.copy_(torch.tensor(np.log(np.maximum(np.asarray(env, np.float32), 1e-8)).reshape(-1, 3),
                                           device=dev))
    rows, images = [], {}
    with torch.no_grad():
        pref = light.prefiltered()
        for v in test_views:
            h, w = v["image"].shape[:2]
            img, cov, m = render_batch(scene, mat, light, *view_tensors([v], dev), w, h, pref)
            photo, mask = v["image"] / 255.0, v["mask"]
            base = torch.where(cov[0, ..., None], m[0, ..., :3], torch.ones_like(m[0, ..., :3])).cpu().numpy()
            matched, gain = gain_match(base, photo, mask)
            renders = {"base_color_unlit": base, "base_color_matched": matched}
            if env is not None:
                renders["relit"] = img[0].cpu().numpy()
            row = {"name": v["name"], "gain": gain}
            for key, image in renders.items():
                row[key] = {"psnr": masked_psnr(image, photo, mask), "lpips": masked_lpips(lpips_metric, image, photo, mask)}
            rows.append(row)
            if keep_images:
                images[v["name"]] = {"photo": photo, "mask": mask, **renders}
    mean = lambda key: {s: float(np.mean([r[key][s] for r in rows])) for s in ("psnr", "lpips")}
    out = {"relit": mean("relit") if env is not None else None, "base_color_unlit": mean("base_color_unlit"),
           "base_color_matched": mean("base_color_matched"), "views": rows}
    if keep_images:
        out["images"] = images
    return out

## Four runs

You score the bake and fit the same object three ways. Every row is scored on the held-out frames, which no fit sees. The `bake` row scores the stage 4 maps with no light: the base color alone (unlit), and again after one exposure gain per frame (gain-matched). It uses stage 6's score functions but this section's renderer and the maps resampled to 1024 px, so its numbers come close to stage 6's bake rows without matching them. The `light_only` run keeps the maps fixed and learns only the light, which shows how much of the gap a light explains with Marigold-IID's maps. The `refined` run learns the maps and the light together with every regularizer. The `no_pull` run repeats it with the pull weights at zero, and the difference between `no_pull` and `refined` is the scale ambiguity in numbers.

Each run reports two scores, masked PSNR and LPIPS as in stage 6. The relit score compares the render under the learned light with the photo. The gain-matched score compares the base color alone, so that the bake and the refined maps are compared on equal terms. The albedo, roughness and metallic means are taken over the chart texels, the albedo in sRGB, and the light mean is the mean linear radiance of the environment map. Steps to converge is the first evaluation, taken every 100 steps, whose held-out relit PSNR is at or above the final value minus 0.1 dB. Training time leaves out the evaluations.

In [ ]:
import warnings

import lpips
import torch

SETTINGS = {"texture": 1024, "data_factor": 2, "steps": 1000, "batch": 4, "env_hw": [32, 64], "env_init": 0.6,
            "lr_tex": 0.01, "lr_env": 0.03, "lr_decay": 0.1, "roughness_range": [0.08, 1.0], "w_tv": [0.5, 1.0, 1.0],
            "w_pull": [0.02, 0.01, 0.01], "w_white": 0.01, "mask_erode_px": 2, "seed": 0, "eval_every": 100}
SIZE = SETTINGS["texture"]
train_views, left_out = scoreable_views(load_views(STAGE1, train, masks, SETTINGS["data_factor"]), drifting)
test_views, left_out_test = scoreable_views(load_views(STAGE1, test, masks, SETTINGS["data_factor"]), drifting)
for name, reason in left_out + left_out_test:
    print(f"{name} left out: {reason}")
h, w = train_views[0]["image"].shape[:2]
print(f"{len(train_views)} training and {len(test_views)} held-out views at {w} x {h} pixels")

MESH_PATH = STAGE4 / "mesh_uv.obj"
mesh_uv = load_mesh_uv(MESH_PATH)
read = lambda name: cv2.imread(str(STAGE4 / f"{name}.png"), cv2.IMREAD_UNCHANGED)
maps0 = {"base_color": cv2.cvtColor(read("base_color"), cv2.COLOR_BGR2RGB) / 255.0,
         "roughness": read("roughness") / 255.0, "metallic": read("metallic") / 255.0}
bake = {k: cv2.resize(np.asarray(a, np.float32), (SIZE, SIZE), interpolation=cv2.INTER_AREA) for k, a in maps0.items()}
scene = Scene(mesh_uv)
chart = scene.chart_mask(SIZE).cpu().numpy()
with warnings.catch_warnings():  # lpips 0.1.4 passes torchvision an argument it has deprecated
    warnings.simplefilter("ignore", UserWarning)
    lpips_metric = lpips.LPIPS(net="alex", verbose=False).cuda().eval()
print(f"stage 4 mesh {len(mesh_uv['faces']):,} triangles; maps of {maps0['roughness'].shape[0]} px fitted at "
      f"{SIZE} px, {chart.mean():.1%} of the texture inside a chart")

In [ ]:
torch.cuda.reset_peak_memory_stats()
evals = {"bake": evaluate_refined(scene, bake, None, test_views, lpips_metric, keep_images=True)}
runs = {"bake": {"base_color_unlit": evals["bake"]["base_color_unlit"],
                 "base_color_matched": evals["bake"]["base_color_matched"], **chart_means(bake, chart),
                 "light_mean": None, "steps_to_converge": None, "seconds": None}}
results = {}
fit = {k: SETTINGS[k] for k in ("steps", "batch", "lr_tex", "lr_env", "w_tv", "w_pull", "w_white", "eval_every", "seed")}
for name, options in (("light_only", {"light_only": True}), ("refined", {}), ("no_pull", {"w_pull": (0.0, 0.0, 0.0)})):
    r = refine(scene, train_views, test_views, maps0, size=SIZE, env_hw=tuple(SETTINGS["env_hw"]), **{**fit, **options})
    evals[name] = evaluate_refined(scene, r["maps"], r["env"], test_views, lpips_metric, keep_images=name == "refined")
    results[name] = r
    runs[name] = {"relit": evals[name]["relit"], "base_color_matched": evals[name]["base_color_matched"],
                  "albedo_mean_srgb": r["albedo_mean_srgb"], "light_mean": r["light_mean"],
                  "metallic_mean": r["metallic_mean"], "roughness_mean": r["roughness_mean"],
                  "steps_to_converge": r["steps_to_converge"], "seconds": r["seconds"]}
    print(f"{name}: held-out relit PSNR by step "
          + ", ".join(f"{e['step']}: {e['psnr']:.2f}" for e in r["history"]))

In [ ]:
score = lambda s: f"{s['psnr']:.2f} / {s['lpips']:.3f}"
print(f"{'run':<11}{'relit PSNR / LPIPS':<21}{'base color matched':<20}{'albedo mean (sRGB)':<21}"
      f"{'light':>7}{'rough':>7}{'metal':>7}{'steps':>7}{'train s':>9}")
for name, r in runs.items():
    relit = score(r["relit"]) if "relit" in r else score(r["base_color_unlit"]) + " unlit"
    albedo = ", ".join(f"{c:.3f}" for c in r["albedo_mean_srgb"])
    light = "-" if r["light_mean"] is None else f"{r['light_mean']:.3f}"
    seconds = "-" if r["seconds"] is None else f"{r['seconds']:.1f}"
    print(f"{name:<11}{relit:<21}{score(r['base_color_matched']):<20}{albedo:<21}{light:>7}"
          f"{r['roughness_mean']:>7.3f}{r['metallic_mean']:>7.3f}{r['steps_to_converge'] or '-':>7}{seconds:>9}")
print("refined, held-out relit PSNR per frame: "
      + ", ".join(f"{v['name']} {v['relit']['psnr']:.2f}" for v in evals["refined"]["views"]))
print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**20:,.0f} MiB (torch)")

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

fig, (ax, ax_env) = plt.subplots(1, 2, figsize=(15, 3.9), gridspec_kw={"width_ratios": [1, 1.5]})
for name in ("light_only", "refined", "no_pull"):
    hist = results[name]["history"]
    ax.plot([e["step"] for e in hist], [e["psnr"] for e in hist], marker="o", ms=3, label=name)
ax.axhline(runs["bake"]["base_color_unlit"]["psnr"], color="gray", ls=":", label="bake, unlit")
ax.axhline(runs["bake"]["base_color_matched"]["psnr"], color="gray", ls="--", label="bake, gain-matched")
ax.set_xlabel("step")
ax.set_ylabel("held-out PSNR (dB)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
env = results["refined"]["env"]
ax_env.imshow(linear_to_srgb(env / env.max()), extent=(-180, 180, 180, 0), aspect="auto")
ax_env.set_xlabel("longitude (degrees)")
ax_env.set_ylabel("angle from straight up (degrees)")
ax_env.set_title(f"learned light of the refined run, scaled by its maximum radiance {env.max():.2f}", fontsize=10)
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=80, bbox_inches="tight")
plt.close(fig)
display(IPImage(buf.getvalue()))

In [ ]:
def crop_box(mask, pad=0.12):
    ys, xs = np.nonzero(mask)
    p = int(pad * max(ys.max() - ys.min(), xs.max() - xs.min()))
    return (slice(max(ys.min() - p, 0), min(ys.max() + p, mask.shape[0])),
            slice(max(xs.min() - p, 0), min(xs.max() + p, mask.shape[1])))


by_psnr = sorted(evals["refined"]["views"], key=lambda r: r["relit"]["psnr"])
bake_rows = {r["name"]: r for r in evals["bake"]["views"]}
picks = [by_psnr[0], by_psnr[len(by_psnr) // 2], by_psnr[-1]]
fig, axes = plt.subplots(len(picks), 4, figsize=(15, 3.2 * len(picks)))
for axrow, row, label in zip(axes, picks, ("lowest", "median", "highest")):
    im, im_bake = evals["refined"]["images"][row["name"]], evals["bake"]["images"][row["name"]]
    box = crop_box(im["mask"])
    error = np.where(im["mask"], np.abs(im["relit"] - im["photo"]).mean(-1), 0.0)
    panels = [(f"photo {row['name']} ({label} refined score)", im["photo"], None),
              (f"bake, gain-matched: {bake_rows[row['name']]['base_color_matched']['psnr']:.1f} dB",
               im_bake["base_color_matched"], None),
              (f"refined, relit: {row['relit']['psnr']:.1f} dB", im["relit"], None),
              ("|refined - photo| inside the mask", error, "magma")]
    for ax, (title, img, cmap) in zip(axrow, panels):
        ax.imshow(np.clip(img[box], 0, 1), cmap=cmap, vmin=0, vmax=0.5 if cmap else 1)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=80, bbox_inches="tight")
plt.close(fig)
display(IPImage(buf.getvalue()))

## What the model leaves out

The relit renders still sit several decibels under the stage 3 splat. Three kinds of error make up the gap: light that the model cannot express, a change of exposure across a whole frame, which one fixed light cannot follow, and a band along the silhouette where the mesh outline and the stage 2 mask disagree. A distant environment with no visibility test gives every point the full sky. A point in a crease or under an overhang is lit as brightly as one on top, and the object casts no shadow on itself. Light that bounces from one part of the object onto another is missing as well.

The table is the largest such effect. It sits under the object and reflects the room's light back up. The fit can represent that only as light from the lower half of the environment map, the same for every point, while the real bounce is strong near the base and fades with height. The fit has one other place to put such effects, the albedo. A contact shadow near the base, or a dark crease, can end up painted into the base color. That raises the gain-matched score without describing the surface any better, and without a ground-truth albedo the scores cannot tell the two apart.

## Refined mesh in the viewer

The viewer shows the stage 4 mesh with the refined base color in the world frame of stage 1, and the learned light as a latitude-longitude image with its top row straight up. The `heldout_view` timeline steps through the held-out frames. For each one it shows the photo, the bake (base color, gain-matched) and the refined render under the learned light. Scrub it and look for the places where the refined render still departs from the photo.

In [ ]:
def log_stage4b(mesh_uv, result, bake_images, refined_images):
    texture = np.round(np.clip(result["maps"]["base_color"], 0, 1) * 255).astype(np.uint8)
    rr.log("world/mesh_refined", rr.Mesh3D(vertex_positions=mesh_uv["vertices"], triangle_indices=mesh_uv["faces"],
                                           vertex_normals=mesh_uv["normals"], vertex_texcoords=mesh_uv["uvs"],
                                           albedo_texture=texture), static=True)
    env = result["env"]
    light = np.round(linear_to_srgb(env / env.max()) * 255).astype(np.uint8)  # row 0 straight up
    rr.log("light/environment", rr.Image(cv2.resize(light, (512, 256), interpolation=cv2.INTER_NEAREST)), static=True)
    for i, name in enumerate(sorted(refined_images)):
        rr.set_time("heldout_view", sequence=i)
        shown = {"photo": refined_images[name]["photo"], "bake": bake_images[name]["base_color_matched"],
                 "refined": refined_images[name]["relit"]}
        for key, img in shown.items():
            img = np.round(np.clip(img, 0, 1) * 255).astype(np.uint8)
            k = 480 / img.shape[1]
            img = cv2.resize(img, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
            rr.log(f"heldout/{key}", rr.EncodedImage(contents=cv2.imencode(".jpg", img[..., ::-1])[1].tobytes(),
                                                     media_type="image/jpeg"))


log_stage4b(mesh_uv, results["refined"], evals["bake"]["images"], evals["refined"]["images"])

In [ ]:
show_rerun()

## Check

The refined maps must lie in $[0, 1]$, with roughness at or above 0.08, and at least three held-out frames must have usable masks. On the reference carton, the `refined` run must also come within 1 dB of the course's reference result, 21.46 dB relit; on another reference object that comparison is printed as a warning. On your own object, a `refined` run that scores below `light_only` on the held-out frames prints a warning: changing the maps then fitted the training photos at the expense of the held-out ones.

When the check passes, it writes `stage4b/`. The folder holds the refined maps on the stage 4 UV layout, the learned light as `env.npy`, and `refine.json` with the four runs and the hash of the `stage4/mesh_uv.obj` they were fitted to. If you rerun stage 4 later, the hash no longer matches, and stage 6 asks you to rerun this section.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np


def write_stage4b(out: Path, result: dict, record: dict) -> None:
    """Write stage4b/ in one step: the refined run's maps as 8-bit PNGs on the stage 4 UV layout
    (base color sRGB), env.npy (float32 (h, w, 3) linear radiance, row 0 at +Z) and refine.json."""
    with atomic_dir(out) as d:
        to8 = lambda a: np.round(np.clip(a, 0, 1) * 255).astype(np.uint8)
        maps = result["maps"]
        ok = [cv2.imwrite(str(d / "base_color.png"), to8(maps["base_color"])[..., ::-1]),
              cv2.imwrite(str(d / "roughness.png"), to8(maps["roughness"])),
              cv2.imwrite(str(d / "metallic.png"), to8(maps["metallic"]))]
        if not all(ok):
            raise RuntimeError("a texture could not be written; stage4b/ was not written")
        np.save(d / "env.npy", np.asarray(result["env"], dtype=np.float32))
        (d / "refine.json").write_text(json.dumps(record, indent=1))

In [ ]:
import math

import numpy as np

REFERENCE_RELIT_PSNR = 21.46  # dB: the carton's refined run, held-out relit, at the course settings (a fixed number)
MIN_HELDOUT = 3  # scoreable held-out frames needed to score a run
ROUGHNESS_FLOOR = 0.08
RELIT_GATE_OBJECT = "boxed_beverage_001"  # the reference object REFERENCE_RELIT_PSNR was measured on


def stage4b_problems(record: dict, on_ref: bool, maps: dict | None = None, gate_relit: bool = True) -> list[str]:
    """Problems with a stage 4b run: failures first, then warnings, which start with "warning:".

    record is refine.json's content and maps the refined run's maps (float or read back from the
    8-bit PNGs). Failures: too few scoreable held-out frames; a map outside [0, 1] or not finite;
    roughness below the 0.08 floor; on the reference only, a refined relit PSNR more than 1 dB
    under REFERENCE_RELIT_PSNR. A refined run that scores below the light-only run warns on your
    own object.

    gate_relit: the relit PSNR gate fails only on the reference object it was measured on
    (RELIT_GATE_OBJECT); on another reference object it is a warning.
    """
    failures, warnings = [], []
    n = record.get("heldout_frames", 0)
    if n < MIN_HELDOUT:
        failures.append(f"only {n} scoreable held-out frame(s), at least {MIN_HELDOUT} are needed: check the stage 2 "
                        "masks of the held-out frames")
    if maps is not None:
        for k in ("base_color", "roughness", "metallic"):
            a = np.asarray(maps[k], dtype=float)
            if not np.isfinite(a).all() or a.min() < 0 or a.max() > 1:
                failures.append(f"{k} has values outside [0, 1]")
        rough = np.asarray(maps["roughness"], dtype=float)
        if np.isfinite(rough).all() and rough.min() < ROUGHNESS_FLOOR - 0.5 / 255:  # 8-bit rounding allowed
            failures.append(f"roughness falls to {rough.min():.3f}, below the {ROUGHNESS_FLOOR} floor: check the clamp "
                            "after each optimizer step")
    runs = record.get("runs", {})
    refined = ((runs.get("refined") or {}).get("relit") or {}).get("psnr")
    light_only = ((runs.get("light_only") or {}).get("relit") or {}).get("psnr")
    if n >= MIN_HELDOUT and (refined is None or not math.isfinite(refined)):
        failures.append("the record has no finite relit PSNR for the refined run")
    elif refined is not None and on_ref and refined < REFERENCE_RELIT_PSNR - 1.0:
        message = (f"on the reference the refined run scores {refined:.2f} dB relit, more than 1 dB under the "
                   f"reference run's {REFERENCE_RELIT_PSNR:.2f} dB")
        if gate_relit:
            failures.append(message + ": check the shading and the loss")
        else:
            warnings.append(f"warning: {message}, which was measured on {RELIT_GATE_OBJECT}, so on this object it "
                            "is printed, not enforced")
    if (not on_ref and refined is not None and light_only is not None and math.isfinite(refined)
            and refined < light_only):
        warnings.append(f"warning: the refined run ({refined:.2f} dB) scores below the light-only run "
                        f"({light_only:.2f} dB) on the held-out frames, so changing the maps made them fit the "
                        "training photos at the cost of the held-out ones")
    return failures + warnings

In [ ]:
import hashlib

record = {"settings": SETTINGS, "mesh_uv_sha256": hashlib.sha256(MESH_PATH.read_bytes()).hexdigest(),
          "train_frames": len(train_views), "heldout_frames": len(test_views), "runs": runs}
refined = results["refined"]["maps"]
print("refined maps: " + ", ".join(f"{k} {refined[k].min():.3f} to {refined[k].max():.3f}"
                                   for k in ("base_color", "roughness", "metallic")))
problems = stage4b_problems(record, REFERENCE, maps=refined,
                            gate_relit=bool(SCAN_OBJECT) and SCAN_OBJECT.endswith(RELIT_GATE_OBJECT))
failures = [p for p in problems if not p.startswith("warning:")]
for p in problems:
    print(p)
print(f"stage time {(time.time() - T0) / 60:.1f} min")
assert not failures, f"{len(failures)} problem(s) above; nothing was written"
write_stage4b(PROJECT / "stage4b", results["refined"], record)
if REFERENCE:
    (PROJECT / "stage4b" / ".reference").touch()  # a later stage then knows these maps are the reference's
print("passed")

### Question 1. Split-sum shading

Explain when the product of the two split-sum factors equals the integral it replaces, and use that to say which combination of lobe width and light makes the approximation err most. Then explain what a second series of photos, taken with the lamp moved to another known position, would tell the fit that the moving camera cannot. Point to the held-out frame where the `refined` render departs most from the photo, and say which effect explains the difference.

*Write your answer here.*

### Question 2. One scale for albedo and light

This section shows the trade for a diffuse surface and a dielectric's highlight. Work out what scaling the linear albedo by $c$ and the light by $1/c$ does to the specular term of a metal ($m = 1$, so $F_0 = k_d$), using $F_0 A + B$, and say at which roughness and viewing angle the trade is furthest from exact. Then say what in a capture would fix the scale physically. Compare the `refined` and `no_pull` runs in `refine.json`: albedo mean, light mean and relit PSNR.

*Write your answer here.*

### Question 3. Metallic under one light

Explain why one fixed light leaves metallic free. Compare a metal, whose highlight carries a tinted $F_0$, with a dielectric that has a darker diffuse color and a broad specular lobe, and say which views would tell them apart. On the course's hammer, metallic after refinement stayed at about 0.50 on the steel head and 0.44 on the wooden handle, and weaker regularizers raised both parts together at the same held-out score. Give the metallic mean of your object before and after refinement, and say whether stage 5's two-density split, which counts a convex piece as metal when its mean metallic exceeds 0.5, would change.

*Write your answer here.*